In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/home-credit-default-risk/bureau_balance.csv
/kaggle/input/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/home-credit-default-risk/previous_application.csv
/kaggle/input/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/home-credit-default-risk/installments_payments.csv
/kaggle/input/home-credit-default-risk/bureau.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/sample_submission.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/application_train.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/home-credit-default-risk/home-credit-default-risk/application_test.csv


# Notebook 02 — Leakage-Safe Modeling Pipeline

This notebook rebuilds the credit-risk model from scratch using
**leakage-safe and bias-aware practices**.

In contrast to Notebook 01, we explicitly enforce:
- Train–test separation **before** preprocessing
- Proper handling of categorical features
- Pipelines to prevent preprocessing and CV leakage
- Metrics aligned with class imbalance
- Explicit legitimacy boundaries

## Objective
To estimate **honest and reproducible model performance** for predicting
loan default using only information available at application time.

## Key Contrast with Notebook 01
- Notebook 01 showed *how models fail silently*
- Notebook 02 shows *how models should be built correctly*



In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score, confusion_matrix


In [4]:
import pandas as pd

data_path = "/kaggle/input/home-credit-default-risk/home-credit-default-risk/application_train.csv"
df = pd.read_csv(data_path)

df.shape


(307511, 122)

# Proper Train/Test Split

In [5]:
# Separate features and target
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# LEAKAGE-SAFE SPLIT
# Split BEFORE any preprocessing or feature selection
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True))


Train shape: (246008, 121)
Test shape: (61503, 121)
Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


### Splitting before preprocessing blocks preprocessing, feature-selection, and CV leakage at the root.

# Identify Numeric vs Categorical Columns (Train-Only)

In [6]:
# Identify feature types using TRAIN data only
# This prevents test-only categories from influencing preprocessing design

cat_cols = X_train.select_dtypes(include="object").columns.tolist()
num_cols = X_train.select_dtypes(exclude="object").columns.tolist()

print("Number of numeric features:", len(num_cols))
print("Number of categorical features:", len(cat_cols))


Number of numeric features: 105
Number of categorical features: 16


# Numeric Preprocessing Pipeline (Leakage-Safe)

In [7]:
# Numeric preprocessing pipeline
# Imputation and scaling will be fit ONLY on training data
# Prevents preprocessing and CV leakage

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

num_pipeline



Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

### Numeric statistics (median, mean, std) are learned only from training data via the pipeline, fixing preprocessing leakage.

# Categorical Preprocessing Pipeline (Leakage-Safe)

In [8]:
# Categorical preprocessing pipeline
#  Impute missing categories using TRAIN data only
#  One-hot encode categories safely
#  handle_unknown="ignore" prevents test-only categories from breaking the model

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

cat_pipeline



Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

### Encoding is fit only on training data and safely handles unseen categories, fixing categorical preprocessing and CV leakage

# Combine Numeric and Categorical Pipelines

In [9]:
# Combine numeric and categorical preprocessing
# Each part is applied to the correct columns only

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)

preprocessor




ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SK_ID_CURR', 'CNT_CHILDREN',
                                  'AMT_INCOME_TOTAL', 'AMT_CREDIT',
                                  'AMT_ANNUITY', 'AMT_GOODS_PRICE',
                                  'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH',
                                  'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
                                  'DAYS_ID_PUBLISH', 'OWN_CAR_AGE',
                                  'FLAG_MOBIL', 'FLA...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['NAME_CONTRACT_TYPE', 'CODE_GENDER',
                                  'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
                                  'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE',
                                  'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
                                  'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE',
                                  'WEEKDAY_APPR_PROCESS_START',
                                  'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE',
                                  'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE',
                                  'EMERGENCYSTATE_MODE'])])

# Build Full Leakage-Safe Model Pipeline

In [10]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])


## Train the Leakage-Safe Model

In [11]:
model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SK_ID_CURR', 'CNT_CHILDREN',
                                                   'AMT_INCOME_TOTAL',
                                                   'AMT_CREDIT', 'AMT_ANNUITY',
                                                   'AMT_GOODS_PRICE',
                                                   'REGION_POPULATION_RELATIVE',
                                                   'DAYS_BIRTH',
                                                   'DAYS_EMPLOYED',
                                                   'DAYS_REGISTRATION',
                                                   'DAYS_ID_PUBLISH'...
                                                   'CODE_GENDER',
                                                   'FLAG_OWN_CAR',
                                                   'FLAG_OWN_REALTY',
                                                   'NAME_TYPE_SUITE',
                                                   'NAME_INCOME_TYPE',
                                                   'NAME_EDUCATION_TYPE',
                                                   'NAME_FAMILY_STATUS',
                                                   'NAME_HOUSING_TYPE',
                                                   'OCCUPATION_TYPE',
                                                   'WEEKDAY_APPR_PROCESS_START',
                                                   'ORGANIZATION_TYPE',
                                                   'FONDKAPREMONT_MODE',
                                                   'HOUSETYPE_MODE',
                                                   'WALLSMATERIAL_MODE',
                                                   'EMERGENCYSTATE_MODE'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

## Proper Evaluation (ROC-AUC + Confusion Matrix)

In [12]:
# Predict probabilities on the TEST set
# Test data is used ONLY here, after full training is complete

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

# Evaluation metrics suitable for imbalanced data
roc_auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print("ROC-AUC (Leakage-Safe):", roc_auc)
print("Confusion Matrix:")
print(cm)


ROC-AUC (Leakage-Safe): 0.7482641428198243
Confusion Matrix:
[[56490    48]
 [ 4903    62]]


## Recall & Classification Report

In [13]:
from sklearn.metrics import recall_score, classification_report

# Recall for defaulters (TARGET = 1)
recall = recall_score(y_test, y_pred)

print("Recall (Defaulters):", recall)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Recall (Defaulters): 0.012487411883182276

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.56      0.01      0.02      4965

    accuracy                           0.92     61503
   macro avg       0.74      0.51      0.49     61503
weighted avg       0.89      0.92      0.88     61503



## Interpretation of Results

- Overall accuracy is still high (~92%), but this is expected due to class imbalance.
- Recall for defaulters is very low (~1.2%), meaning the model catches very few actual defaulters.
- This low recall is NOT a bug — it reflects the true difficulty of the problem.
- Unlike Notebook 01, these results are honest because no leakage is present.

Key difference from Notebook 01:
- Notebook 01 showed high accuracy due to leakage and bias.
- Notebook 02 shows similar accuracy, but now it is trustworthy and realistic.


## Leakages Fixed in This Notebook

The following issues from Notebook 01 are fully fixed:

- Preprocessing leakage  
  → All imputation and scaling are done after train–test split inside a pipeline.

- Feature selection leakage  
  → No feature selection is done using full data.

- Cross-validation leakage  
  → Pipelines ensure each fold learns preprocessing independently.

- Identifier leakage  
  → No ID-based manual feature handling is performed.

- Metric bias  
  → ROC-AUC and recall are used instead of accuracy alone.

- Design-decision leakage  
  → All decisions are based only on training data.


# Temporal Leakage Analysis
* Random train–test splits can violate causality in time-dependent problems like credit risk. Even with leakage-safe preprocessing, random splits may introduce optimistic bias. We therefore compare a random split against a pseudo-temporal split using a timeline proxy.


# Temporal Train/Test Split (Evaluation Stress Test)

In [14]:
# ---- TEMPORAL EVALUATION (NO LEAKAGE) ----
# We do NOT retrain the pipeline design using test data.
# We only change HOW we split data to test robustness.

# Use a time proxy to simulate real-world ordering
# Smaller DAYS_ID_PUBLISH = older records
df_sorted = df.sort_values("DAYS_ID_PUBLISH").reset_index(drop=True)

X_temp = df_sorted.drop(columns=["TARGET"])
y_temp = df_sorted["TARGET"]

# 80% past → 20% future
split_idx = int(0.8 * len(df_sorted))

X_train_t = X_temp.iloc[:split_idx]
X_test_t  = X_temp.iloc[split_idx:]
y_train_t = y_temp.iloc[:split_idx]
y_test_t  = y_temp.iloc[split_idx:]

print("Temporal Train shape:", X_train_t.shape)
print("Temporal Test shape:", X_test_t.shape)



Temporal Train shape: (246008, 121)
Temporal Test shape: (61503, 121)


## Train & Evaluate on Temporal Split

In [15]:
# Train the SAME leakage-safe pipeline on past data
model.fit(X_train_t, y_train_t)

# Evaluate on future data
y_proba_t = model.predict_proba(X_test_t)[:, 1]
y_pred_t = model.predict(X_test_t)

roc_auc_temporal = roc_auc_score(y_test_t, y_proba_t)
recall_temporal = recall_score(y_test_t, y_pred_t)
cm_temporal = confusion_matrix(y_test_t, y_pred_t)

print("ROC-AUC (Temporal Split):", roc_auc_temporal)
print("Recall (Temporal Split):", recall_temporal)
print("Confusion Matrix (Temporal Split):")
print(cm_temporal)



ROC-AUC (Temporal Split): 0.7374287411637438
Recall (Temporal Split): 0.017897819720143184
Confusion Matrix (Temporal Split):
[[55231   126]
 [ 6036   110]]


### Temporal vs Random Split (Result)

The pseudo-temporal split yields a lower ROC-AUC than the random split.
This indicates optimistic bias under random splitting, where future
distributional information can influence training despite leakage-safe preprocessing.


## Temporal Evaluation & Key Conclusions

- ROC-AUC drops to ~0.74 when evaluated on future data.
- Recall remains low, showing the true difficulty of detecting defaulters early.
- Temporal splitting gives more realistic results than random splitting.
- Performance drops confirm that leakage has been removed.

### Final Takeaway
Leakage-free pipelines produce weaker but honest results.
Correct evaluation matters more than model choice.
